In [ ]:
!pip install transformers peft accelerate bitsandbytes datasets \
             sentence-transformers faiss-cpu gradio trl -q

print("✅ Kurulum tamamlandı!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.4 MB/s eta 0:00:00
✅ Kurulum tamamlandı!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import glob
import os

# Zip dosyasını çıkart (Colab'a yüklediğini varsayıyorum)
import zipfile
with zipfile.ZipFile('archive (1).zip', 'r') as z: # Dosya adı düzeltildi
    z.extractall('clinical_notes/')

# Tüm JSON dosyalarını oku
all_notes = []
for filepath in sorted(glob.glob('clinical_notes/*.json')):
    with open(filepath, 'r') as f:
        data = json.load(f)
        all_notes.append(data)

print(f"Toplam dosya sayısı: {len(all_notes)}")
print(f"\n--- İlk kaydın yapısı ---")
print("Kolonlar:", list(all_notes[0].keys()))
print("\n--- Hasta bilgisi ---")
print(all_notes[0]['sample'])
print("\n--- Notun ilk 500 karakteri ---")
print(all_notes[0]['note'][:500])

FileNotFoundError: [Errno 2] No such file or directory: 'archive (1).zip'

In [ ]:
import json
import glob

# Veriyi fine-tuning formatına dönüştür
training_data = []

for filepath in sorted(glob.glob('/content/clinical_notes/*.json')):
    with open(filepath, 'r') as f:
        data = json.load(f)

    sample = data['sample']
    note = data['note']

    # Çok uzun notları kırp (LLaMA'nın token limiti var)
    note_trimmed = note[:3000]

    # Instruction formatı oluştur
    instruction = f"""You are a medical doctor. Write a detailed clinical discharge note (epikriz) for the following patient.

Patient Information:
- Age: {sample['age']}
- Gender: {sample['gender']}
- Disease: {sample['disease']}
- Stage: {sample.get('stage', 'unknown')}

Write a structured clinical note with patient information, visit details, diagnosis, treatment, and discharge plan."""

    training_data.append({
        "instruction": instruction,
        "output": note_trimmed
    })

# Kaydet
with open('/content/training_data.json', 'w') as f:
    json.dump(training_data, f, ensure_ascii=False, indent=2)

print(f"✅ {len(training_data)} eğitim örneği hazırlandı!")
print("\n--- Örnek instruction ---")
print(training_data[0]['instruction'])
print("\n--- Çıktının ilk 300 karakteri ---")
print(training_data[0]['output'][:300])

In [ ]:
from huggingface_hub import login

import os
token = os.environ.get("HF_TOKEN")  # Çevre değişkeninden oku

print("✅ HuggingFace'e giriş yapıldı!")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization ayarı (Colab RAM'ini korur)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "meta-llama/Llama-3.2-3B-Instruct"

print("⏳ Model indiriliyor... (10-15 dk sürebilir)")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Model yüklendi!")

# LoRA ayarları
lora_config = LoraConfig(
    r=8,                        # LoRA rank (düşük = hızlı, az GPU)
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Kaç parametre eğitilecek göster
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA hazır!")
print(f"Eğitilecek parametre: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
!pip install transformers peft accelerate bitsandbytes>=0.46.1 datasets \
sentence-transformers faiss-cpu gradio trl -q

print("✅ Kurulum tamamlandı!")

In [ ]:
!pip install --upgrade --force-reinstall pyarrow datasets -q
print("✅ pyarrow ve datasets yeniden yüklendi.")

In [ ]:
import json
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Eğitim verisini yükle
with open('/content/training_data.json', 'r') as f:
    raw_data = json.load(f)

# HuggingFace Dataset formatına çevir
def format_sample(sample):
    return {
        "text": f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{sample['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{sample['output']}<|eot_id|>"""
    }

formatted = [format_sample(s) for s in raw_data]
dataset = Dataset.from_list(formatted)

print(f"✅ Dataset hazır: {len(dataset)} örnek")

# Eğitim ayarları
sft_config = SFTConfig(
    output_dir="/content/llama-epikriz",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True, # fp16 yerine bf16 kullan
    logging_steps=10,
    save_steps=50,
    report_to="none"
)

# Trainer oluştur
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config
)

print("\n⏳ Eğitim başlıyor... (20-30 dk sürebilir)")
trainer.train()
print("\n✅ Eğitim tamamlandı!")

## Eğitilmiş Modelin Kaydedilmesi

Eğitim tamamlandıktan sonra, yalnızca LoRA adaptörlerini kaydedebilirsiniz. Bu, tüm model ağırlıklarını kaydetmekten daha verimlidir.

In [ ]:
output_dir = "/content/llama-3-8b-medical-ft"

# Save LoRA adapters
trainer.save_model(output_dir)

# Save tokenizer
tokenizer.save_pretrained(output_dir)

print(f"✅ Model ve tokenizer '{output_dir}' dizinine kaydedildi!")

## Kaydedilmiş Modelin Yüklenmesi

Yeni bir oturumda veya başka bir yerde kaydedilmiş modeli kullanmak için, önce temel modeli ve ardından LoRA adaptörlerini yüklemeniz gerekir.

In [ ]:
!pip install --upgrade transformers --quiet
!pip install --upgrade accelerate --quiet

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Kayıtlı modelin dizini
output_dir = "/content/llama-3-8b-medical-ft"

# Temel modeli ve tokenizer'ı yükle
model_name = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("⏳ Temel model yükleniyor...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    # Hata: Gated repo erişimi için Hugging Face token'ı gerekiyor.
    # Lütfen https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct adresinden erişim isteyin
    # ve token'ınızı Colab Secrets'e 'HF_TOKEN' olarak ekleyip runtime'ı yeniden başlatın.
    token=True # Token'ı açıkça geçirin
)

print("⏳ Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(output_dir)
tokenizer.pad_token = tokenizer.eos_token

# PEFT adaptörlerini temel modele yükle
print("⏳ PEFT adaptörleri yükleniyor...")
model = PeftModel.from_pretrained(base_model, output_dir)

# Modeli değerlendirme moduna ayarla
model.eval()

print("✅ Fine-tuned model başarıyla yüklendi!")

## Yüklenen Modeli Test Etme

Modeli yükledikten sonra, test etmek için daha önce kullandığınız inference kodunu tekrar çalıştırabilirsiniz.

In [ ]:
from transformers import pipeline

test_input = """You are a medical doctor. Write a detailed clinical discharge note (epikriz) for the following patient.

Patient Information:
- Age: 65 years old
- Gender: female
- Disease: Type 2 diabetes mellitus
- Stage: moderate

Write a structured clinical note with patient information, visit details, diagnosis, treatment, and discharge plan."""

prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{test_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Sadece asistan cevabını göster
assistant_response = response.split("assistant")[-1].strip()
print(assistant_response)

In [ ]:
from transformers import pipeline

# Fine-tuned modeli test için ayarla
model.eval()

test_input = """You are a medical doctor. Write a detailed clinical discharge note (epikriz) for the following patient.

Patient Information:
- Age: 65 years old
- Gender: female
- Disease: Type 2 diabetes mellitus
- Stage: moderate

Write a structured clinical note with patient information, visit details, diagnosis, treatment, and discharge plan."""

prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{test_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Sadece asistan cevabını göster
assistant_response = response.split("assistant")[-1].strip()
print(assistant_response)

**ESKI ARAYUZ** (aşağıda rag entegresiyle yenilenmiş hali var)

In [ ]:
import gradio as gr
import torch

def generate_epikriz(age, gender, disease, stage):
    test_input = f"""You are a medical doctor. Write a detailed clinical discharge note (epikriz) for the following patient.

Patient Information:
- Age: {age} years old
- Gender: {gender}
- Disease: {disease}
- Stage: {stage}

Write a structured clinical note with patient information, visit details, diagnosis, treatment, and discharge plan."""

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{test_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("assistant")[-1].strip()

# Arayüz
demo = gr.Interface(
    fn=generate_epikriz,
    inputs=[
        gr.Number(label="Yaş", value=45),
        gr.Dropdown(["male", "female"], label="Cinsiyet", value="male"),
        gr.Textbox(label="Hastalık", placeholder="örn: Type 2 diabetes mellitus"),
        gr.Dropdown(["early", "moderate", "advanced"], label="Evre", value="moderate")
    ],
    outputs=gr.Textbox(label="Üretilen Epikriz", lines=30),
    title="🏥 Sentetik Epikriz Üretici",
    description="Hasta bilgilerini girin, yapay zeka klinik rapor oluştursun."
)

demo.launch(share=True)

In [ ]:
!pip install rouge-score -q

from rouge_score import rouge_scorer
import json

# Referans notları yükle
with open('/content/training_data.json', 'r') as f:
    raw_data = json.load(f)

# Test için 5 farklı hasta üret ve karşılaştır
test_cases = [
    {"age": "45", "gender": "male", "disease": "hypertension", "stage": "moderate"},
    {"age": "60", "gender": "female", "disease": "Type 2 diabetes mellitus", "stage": "advanced"},
    {"age": "35", "gender": "male", "disease": "pneumonia", "stage": "early"},
    {"age": "70", "gender": "female", "disease": "heart failure", "stage": "advanced"},
    {"age": "50", "gender": "male", "disease": "chronic kidney disease", "stage": "moderate"},
]

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

all_scores = []

for i, case in enumerate(test_cases):
    # Model ile epikriz üret
    test_input = f"""You are a medical doctor. Write a detailed clinical discharge note for the following patient.

Patient Information:
- Age: {case['age']} years old
- Gender: {case['gender']}
- Disease: {case['disease']}
- Stage: {case['stage']}

Write a structured clinical note with patient information, visit details, diagnosis, treatment, and discharge plan."""

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{test_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated = generated.split("assistant")[-1].strip()

    # En benzer referans notu bul
    reference = raw_data[i]['output']

    # ROUGE hesapla
    scores = scorer.score(reference, generated)
    all_scores.append(scores)

    print(f"Hasta {i+1} ({case['disease']}):")
    print(f"  ROUGE-1: {scores['rouge1'].fmeasure:.4f}")
    print(f"  ROUGE-2: {scores['rouge2'].fmeasure:.4f}")
    print(f"  ROUGE-L: {scores['rougeL'].fmeasure:.4f}")
    print()

# Ortalama skorlar
avg_r1 = sum(s['rouge1'].fmeasure for s in all_scores) / len(all_scores)
avg_r2 = sum(s['rouge2'].fmeasure for s in all_scores) / len(all_scores)
avg_rL = sum(s['rougeL'].fmeasure for s in all_scores) / len(all_scores)

print("=" * 40)
print(f"ORTALAMA ROUGE-1: {avg_r1:.4f}")
print(f"ORTALAMA ROUGE-2: {avg_r2:.4f}")
print(f"ORTALAMA ROUGE-L: {avg_rL:.4f}")

1. Kütüphaneleri yükle
2. Drive'ı bağla
3. Modeli Drive'dan yükle


In [ ]:
# Fine-tuned modeli kaydet
model.save_pretrained("/content/llama-epikriz-final")
tokenizer.save_pretrained("/content/llama-epikriz-final")
print("✅ Model kaydedildi!")

# Google Drive'a yükle (kalıcı olsun)
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree("/content/llama-epikriz-final",
                "/content/drive/MyDrive/llama-epikriz-final")
print("✅ Google Drive'a yüklendi!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Skorlar
diseases = ['Hypertension', 'Diabetes\nType 2', 'Pneumonia', 'Heart\nFailure', 'Chronic\nKidney']
rouge1 = [0.1355, 0.4047, 0.5303, 0.3056, 0.3770]
rouge2 = [0.0704, 0.1265, 0.2614, 0.1244, 0.1418]
rougeL = [0.0888, 0.2023, 0.3455, 0.2179, 0.2150]

x = np.arange(len(diseases))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ROUGE Evaluation Results — Synthetic Epikriz Generation',
             fontsize=14, fontweight='bold')

# --- Sol grafik: Grouped bar chart ---
ax1 = axes[0]
bars1 = ax1.bar(x - width, rouge1, width, label='ROUGE-1', color='#2196F3', alpha=0.85)
bars2 = ax1.bar(x,         rouge2, width, label='ROUGE-2', color='#4CAF50', alpha=0.85)
bars3 = ax1.bar(x + width, rougeL, width, label='ROUGE-L', color='#FF9800', alpha=0.85)

ax1.set_xlabel('Disease Category')
ax1.set_ylabel('F-Measure Score')
ax1.set_title('ROUGE Scores per Disease')
ax1.set_xticks(x)
ax1.set_xticklabels(diseases, fontsize=9)
ax1.set_ylim(0, 0.65)
ax1.legend()
ax1.axhline(y=0.30, color='red', linestyle='--', alpha=0.5, label='0.30 threshold')

# Bar üstüne değer yaz
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)
for bar in bars3:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

# --- Sağ grafik: Ortalama skorlar ---
ax2 = axes[1]
avg_scores = [0.3506, 0.1449, 0.2139]
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
colors = ['#2196F3', '#4CAF50', '#FF9800']

bars = ax2.bar(metrics, avg_scores, color=colors, alpha=0.85, width=0.4)
ax2.set_xlabel('Metric')
ax2.set_ylabel('Average F-Measure Score')
ax2.set_title('Average ROUGE Scores (All Diseases)')
ax2.set_ylim(0, 0.5)

for bar, score in zip(bars, avg_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/rouge_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik kaydedildi: /content/rouge_results.png")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

# Eğitim verisini yükle
with open('/content/training_data.json', 'r') as f:
    training_data = json.load(f)

# Embedding modeli yükle
print("⏳ Embedding modeli yükleniyor...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding modeli hazır!")

# Her notun instruction kısmını vektöre çevir
print("⏳ Vektörler oluşturuluyor...")
instructions = [item['instruction'] for item in training_data]
embeddings = embedder.encode(instructions, show_progress_bar=True)

# FAISS vektör veritabanı oluştur
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"✅ FAISS index hazır! {index.ntotal} vaka yüklendi.")

In [ ]:
def retrieve_similar_cases(query, top_k=3):
    # Sorguyu vektöre çevir
    query_embedding = embedder.encode([query]).astype(np.float32)

    # En benzer 3 vakayı bul
    distances, indices = index.search(query_embedding, top_k)

    similar_cases = []
    for i, idx in enumerate(indices[0]):
        similar_cases.append({
            "rank": i+1,
            "distance": distances[0][i],
            "note_preview": training_data[idx]['output'][:300]
        })
    return similar_cases

def generate_epikriz_with_rag(age, gender, disease, stage):
    # 1. RAG: benzer vakaları bul
    query = f"{age} years old {gender} patient with {disease} {stage} stage"
    similar_cases = retrieve_similar_cases(query, top_k=2)

    # 2. Bağlamı oluştur
    context = "\n\n".join([
        f"Similar case {c['rank']}:\n{c['note_preview']}..."
        for c in similar_cases
    ])

    # 3. RAG destekli prompt
    rag_prompt = f"""You are a medical doctor. Use the similar cases below as reference and write a detailed clinical discharge note for the new patient.

SIMILAR CASES FOR REFERENCE:
{context}

NEW PATIENT:
- Age: {age} years old
- Gender: {gender}
- Disease: {disease}
- Stage: {stage}

Write a structured clinical note with patient information, visit details, diagnosis, treatment, and discharge plan."""

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{rag_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("assistant")[-1].strip()

# Test et
print("⏳ RAG ile epikriz üretiliyor...")
result = generate_epikriz_with_rag("55", "male", "heart failure", "advanced")
print(result)

In [ ]:
import torch
import numpy as np

def calculate_perplexity(text, max_length=512):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss

    perplexity = torch.exp(loss).item()
    return perplexity

# Test metinleri: model çıktısı vs rastgele metin
test_medical = """Patient Information: 65 year old female admitted with Type 2 diabetes mellitus.
Blood glucose 280 mg/dL, HbA1c 9.2%. Started on insulin glargine 20 units.
Discharged with metformin 1000mg twice daily and dietary counseling."""

test_random = """The weather today is sunny with clouds.
The cat sat on the mat eating pizza while watching television programs.
Blue elephants dance slowly near the mountain rivers every Tuesday morning."""

print("⏳ Perplexity hesaplanıyor...")

ppx_medical = calculate_perplexity(test_medical)
ppx_random  = calculate_perplexity(test_random)

# Eğitim verilerinden 5 örnek üzerinde ortalama
ppx_scores = []
for item in training_data[:5]:
    ppx = calculate_perplexity(item['output'])
    ppx_scores.append(ppx)

ppx_avg = np.mean(ppx_scores)

print(f"\n📊 Perplexity Sonuçları:")
print(f"Tıbbi metin (model çıktısı):  {ppx_medical:.2f}")
print(f"Rastgele metin:               {ppx_random:.2f}")
print(f"Eğitim verisi ortalaması:     {ppx_avg:.2f}")
print(f"\n✅ Model tıbbi metni rastgele metinden {ppx_random/ppx_medical:.1f}x daha iyi anlıyor!")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Perplexity Evaluation — Synthetic Epikriz Model',
             fontsize=13, fontweight='bold')

# Sol: Bar chart
ax1 = axes[0]
labels = ['Training data\n(avg)', 'Medical text\n(generated)', 'Random text']
values = [2.42, 8.96, 67.22]
colors = ['#4CAF50', '#2196F3', '#F44336']

bars = ax1.bar(labels, values, color=colors, alpha=0.85, width=0.5)
ax1.set_ylabel('Perplexity (düşük = daha iyi)')
ax1.set_title('Perplexity Karşılaştırması')
ax1.set_ylim(0, 80)

for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

ax1.axhline(y=10, color='gray', linestyle='--', alpha=0.5)
ax1.text(2.35, 11, 'iyi eşik (<10)', fontsize=9, color='gray')

# Sağ: Özet metrik kartı
ax2 = axes[1]
ax2.axis('off')

summary = [
    ['Metrik', 'Değer', 'Yorum'],
    ['PPX — Eğitim verisi', '2.42', '✅ Mükemmel'],
    ['PPX — Tıbbi metin', '8.96', '✅ İyi (<10)'],
    ['PPX — Rastgele metin', '67.22', '❌ Yüksek (beklenen)'],
    ['Tıbbi / Rastgele oranı', '7.5x', '✅ Güçlü fark'],
    ['ROUGE-1 (ort.)', '0.3506', '✅ Kabul edilebilir'],
    ['ROUGE-2 (ort.)', '0.1449', '✅ Normal'],
    ['ROUGE-L (ort.)', '0.2139', '✅ Normal'],
]

table = ax2.table(cellText=summary[1:], colLabels=summary[0],
                  cellLoc='center', loc='center',
                  bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2196F3')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f5f5f5')
    cell.set_edgecolor('#dddddd')

ax2.set_title('Tüm Metrikler Özeti', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('/content/perplexity_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik kaydedildi!")

In [ ]:
import requests
import xml.etree.ElementTree as ET

def fetch_pubmed_context(disease, max_results=3):
    # PubMed'de hastalıkla ilgili makale ara
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": f"{disease} clinical treatment diagnosis",
        "retmax": max_results,
        "retmode": "json"
    }

    search_resp = requests.get(search_url, params=search_params)
    ids = search_resp.json()["esearchresult"]["idlist"]

    if not ids:
        return "No external references found."

    # Bulunan makalelerin özetlerini çek
    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    fetch_params = {
        "db": "pubmed",
        "id": ",".join(ids),
        "rettype": "abstract",
        "retmode": "xml"
    }

    fetch_resp = requests.get(fetch_url, params=fetch_params)
    root = ET.fromstring(fetch_resp.content)

    abstracts = []
    for article in root.findall(".//PubmedArticle"):
        title = article.findtext(".//ArticleTitle", default="")
        abstract = article.findtext(".//AbstractText", default="")
        if abstract:
            abstracts.append(f"Title: {title}\nAbstract: {abstract[:400]}...")

    return "\n\n".join(abstracts) if abstracts else "No abstracts found."

# Test et
print("⏳ PubMed'den veri çekiliyor...")
context = fetch_pubmed_context("heart failure")
print(context[:800])

In [ ]:
def fetch_pubmed_context(disease, stage="", max_results=3):
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    query = f"{disease} clinical treatment guidelines"

    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json",
        "sort": "relevance"
    }

    search_resp = requests.get(search_url, params=search_params)
    ids = search_resp.json()["esearchresult"]["idlist"]

    if not ids:
        return "No external references found."

    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    fetch_params = {
        "db": "pubmed",
        "id": ",".join(ids),
        "rettype": "abstract",
        "retmode": "xml"
    }

    fetch_resp = requests.get(fetch_url, params=fetch_params)
    root = ET.fromstring(fetch_resp.content)

    abstracts = []
    for article in root.findall(".//PubmedArticle"):
        title = article.findtext(".//ArticleTitle", default="")
        abstract = article.findtext(".//AbstractText", default="")
        year = article.findtext(".//PubDate/Year", default="")

        # Sadece abstract boş olanları ele
        if abstract:
            abstracts.append(f"[{year}] {title}\n{abstract[:500]}...")

    return "\n\n---\n\n".join(abstracts) if abstracts else "No abstracts found."

# Test
print("⏳ Test ediliyor...")
result = fetch_pubmed_context("heart failure", "advanced")
print(result[:1000])

In [ ]:
def generate_epikriz_with_pubmed_rag(age, gender, disease, stage):
    print(f"🔍 PubMed'den '{disease}' kılavuzları çekiliyor...")
    pubmed_context = fetch_pubmed_context(disease, stage, max_results=3)

    print(f"📚 Yerel veri tabanından benzer vakalar aranıyor...")
    query = f"{age} years old {gender} patient with {disease} {stage} stage"
    query_embedding = embedder.encode([query]).astype(np.float32)
    distances, indices = index.search(query_embedding, 2)

    local_cases = "\n\n".join([
        f"Similar case {i+1}:\n{training_data[idx]['output'][:300]}..."
        for i, idx in enumerate(indices[0])
    ])

    rag_prompt = f"""You are a medical doctor. Use the clinical guidelines and similar cases below to write a detailed clinical discharge note.

CLINICAL GUIDELINES (PubMed):
{pubmed_context[:800]}

SIMILAR CASES FROM DATABASE:
{local_cases}

NEW PATIENT:
- Age: {age} years old
- Gender: {gender}
- Disease: {disease}
- Stage: {stage}

Write a structured clinical note with patient information, visit details, diagnosis, treatment, and discharge plan."""

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{rag_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("assistant")[-1].strip()

# Gradio arayüzünü güncelle
import gradio as gr

demo = gr.Interface(
    fn=lambda age, gender, disease, stage: generate_epikriz_with_pubmed_rag(
        str(int(age)), gender, disease, stage
    ),
    inputs=[
        gr.Number(label="Yaş", value=45),
        gr.Dropdown(["male", "female"], label="Cinsiyet", value="male"),
        gr.Textbox(label="Hastalık", placeholder="örn: heart failure"),
        gr.Dropdown(["early", "moderate", "advanced"], label="Evre", value="moderate")
    ],
    outputs=gr.Textbox(label="Üretilen Epikriz", lines=35),
    title="🏥 Sentetik Epikriz Üretici",
    description="PubMed klinik kılavuzları + yerel vaka veritabanı destekli üretim."
)

demo.launch(share=True)

**HER BAĞLANTIDA ÇALIŞTIRILACAK OTURUM AÇMA KODLARI!**

In [ ]:
!pip install transformers peft accelerate bitsandbytes datasets \
             sentence-transformers faiss-cpu gradio trl rouge-score -q
print("✅ Kurulum tamamlandı!")

✅ Kurulum tamamlandı!


In [ ]:
from huggingface_hub import login
login(token="YOUR_TOKEN_HERE")
print("✅ HuggingFace girişi tamam!")

✅ HuggingFace girişi tamam!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive bağlandı!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive bağlandı!


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "meta-llama/Llama-3.2-3B-Instruct"
print("⏳ Temel model indiriliyor...")

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/llama-epikriz-final"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=True # Explicitly pass token for gated models
)

model = PeftModel.from_pretrained(
    model,
    "/content/drive/MyDrive/llama-epikriz-final" # Corrected path to saved adapters
)

model.eval()
print("✅ Model hazır!")

⏳ Temel model indiriliyor...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

✅ Model hazır!


In [ ]:
import json, glob, zipfile

with zipfile.ZipFile('/content/archive (1).zip', 'r') as z:
    z.extractall('/content/clinical_notes/')

all_notes = []
for filepath in sorted(glob.glob('/content/clinical_notes/*.json')):
    with open(filepath, 'r') as f:
        all_notes.append(json.load(f))

training_data = []
for data in all_notes:
    sample = data['sample']
    training_data.append({
        "instruction": f"""You are a medical doctor. Write a detailed clinical discharge note for the following patient.
Patient Information:
- Age: {sample['age']}
- Gender: {sample['gender']}
- Disease: {sample['disease']}
- Stage: {sample.get('stage', 'unknown')}
Write a structured clinical note with patient information, visit details, diagnosis, treatment, and discharge plan.""",
        "output": data['note'][:3000]
    })

with open('/content/training_data.json', 'w') as f:
    json.dump(training_data, f, ensure_ascii=False, indent=2)

print(f"✅ {len(training_data)} örnek hazırlandı!")

✅ 100 örnek hazırlandı!


In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

with open('/content/training_data.json', 'r') as f:
    training_data = json.load(f)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
instructions = [item['instruction'] for item in training_data]
embeddings = embedder.encode(instructions, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"✅ RAG hazır! {index.ntotal} vaka yüklendi.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✅ RAG hazır! 100 vaka yüklendi.


In [ ]:
import requests
import xml.etree.ElementTree as ET

def fetch_pubmed_context(disease, stage="", max_results=3):
    search_params = {
        "db": "pubmed",
        "term": f"{disease} clinical treatment guidelines",
        "retmax": max_results,
        "retmode": "json",
        "sort": "relevance"
    }
    ids = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params=search_params
    ).json()["esearchresult"]["idlist"]

    if not ids:
        return "No references found."

    fetch_resp = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
        params={"db": "pubmed", "id": ",".join(ids), "rettype": "abstract", "retmode": "xml"}
    )
    root = ET.fromstring(fetch_resp.content)
    abstracts = []
    for article in root.findall(".//PubmedArticle"):
        title = article.findtext(".//ArticleTitle", default="")
        abstract = article.findtext(".//AbstractText", default="")
        year = article.findtext(".//PubDate/Year", default="")
        if abstract:
            abstracts.append(f"[{year}] {title}\n{abstract[:500]}...")
    return "\n\n---\n\n".join(abstracts) if abstracts else "No abstracts found."

print("✅ PubMed fonksiyonu hazır!")

✅ PubMed fonksiyonu hazır!
